In [1]:
import os
import sys
import gzip
import pickle

import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
# dt_embeds_path = "/home/datalab/nfs/deepfm/data/train_21/embeddings/dt_embeddings_test.pkl.gz"
dt_embeds_path = "/home/datalab/nfs/deepfm/data/train_21/embeddings/07_08_2025_dt_embeddings_test.pkl.gz"
kt_embeds_path = "/home/datalab/nfs/deepfm/data/train_21/embeddings/kt_embeddings_test.pkl.gz"

# dt-шники
with gzip.open(dt_embeds_path, "rb") as f:
    dt_embeds = pickle.load(f)

# kt-шники
with gzip.open(kt_embeds_path, "rb") as f:
    kt_embeds = pickle.load(f)

In [3]:
total_bytes = sum(v.nbytes for v in dt_embeds.values())
print("Length of dt_embds:", len(dt_embeds))
print(f"Size of dt_embeds: {total_bytes / 1024 / 1024:.2f} MB\n")

total_bytes = sum(v.nbytes for v in kt_embeds.values())
print("Length of kt_embds:", len(kt_embeds))
print(f"Size of kt_embeds: {total_bytes / 1024 / 1024:.2f} MB\n")

Length of dt_embds: 15
Size of dt_embeds: 0.01 MB

Length of kt_embds: 2364265
Size of kt_embeds: 1154.43 MB



In [4]:
# БЫЛО
total_bytes = sum(v.nbytes for v in dt_embeds.values())
print("Length of dt_embds:", len(dt_embeds))
print(f"Size of dt_embeds: {total_bytes / 1024 / 1024:.2f} MB\n")

total_bytes = sum(v.nbytes for v in kt_embeds.values())
print("Length of kt_embds:", len(kt_embeds))
print(f"Size of kt_embeds: {total_bytes / 1024 / 1024:.2f} MB\n")

Length of dt_embds: 1606165
Size of dt_embeds: 784.26 MB

Length of kt_embds: 1370811
Size of kt_embeds: 669.34 MB



In [8]:
# from tqdm import tqdm

# item_ids = list(kt_embeds.keys())
# item_vectors = np.vstack([kt_embeds[i] for i in item_ids]).astype("float32")

# dim = item_vectors.shape[1]
# index = faiss.IndexFlatIP(dim)
# index.add(item_vectors)

# top_k = 20
# batch_size = 1000
# user_ids = list(dt_embeds.keys())
# results = {}

# for i in tqdm(range(0, len(user_ids), batch_size), desc="Processing Users"):
#     batch_ids = user_ids[i:i+batch_size]
#     batch_vectors = np.vstack([dt_embeds[u] for u in batch_ids]).astype("float32")
    
#     D, I = index.search(batch_vectors, top_k)
    
#     for j, user_id in enumerate(batch_ids):
#         results[user_id] = [item_ids[k] for k in I[j]]

Processing Users:   0%|          | 0/1607 [00:00<?, ?it/s]

In [4]:
from annoy import AnnoyIndex

dim = 128
top_k = 100
n_trees = 60

item_ids = list(kt_embeds.keys())
item_index_map = {i: item_id for i, item_id in enumerate(item_ids)}
annoy_index = AnnoyIndex(dim, "angular")

for i, item_id in tqdm(enumerate(item_ids)):
    annoy_index.add_item(i, kt_embeds[item_id])
    
annoy_index.build(n_trees)

2364265it [00:39, 59837.62it/s]


True

In [5]:
from tqdm import tqdm

results = {}
user_ids = list(dt_embeds.keys())

for uid in tqdm(user_ids, desc="Annoy search"):
    vec = dt_embeds[uid]
    idxs = annoy_index.get_nns_by_vector(vec, top_k)
    results[uid] = [item_index_map[i] for i in idxs]

Annoy search: 100%|██████████| 15/15 [00:00<00:00, 228.10it/s]


In [6]:
import gzip
import pickle

def save_compressed(path, embeddings):
    with gzip.open(path, "wb") as f:
        pickle.dump(embeddings, f, protocol=pickle.HIGHEST_PROTOCOL)
        
path = "/home/datalab/nfs/deepfm/data/train_21/recommendations/07_08_2025_test_kts_for_dts.pkl.gz"  
save_compressed(path, results)